<a href="https://colab.research.google.com/github/remix0091/Big-Data/blob/main/L2_Introduction_to_Apache_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №2. Введение в Apache Spark
Бондарев Иван 6403

In [1]:
# Установка Java 17 и PySpark
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q pyspark

# Переменные окружения
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Проверка Java
!java -version

# Запуск SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BikeShareAnalysis")
    .master("local[*]")
    .config("spark.ui.port", "4050")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version:", sc.version)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
Spark version: 4.0.2


In [2]:
from google.colab import files
uploaded = files.upload()

In [4]:
from pyspark.sql.functions import col

# Загрузка данных
trips = spark.read.option("header", True).csv("trip.csv")
stations = spark.read.option("header", True).csv("station.csv")

# Приведение типов
trips = trips \
    .withColumn("duration", col("duration").cast("int")) \
    .withColumn("bike_id", col("bike_id").cast("int"))

stations = stations \
    .withColumn("lat", col("lat").cast("double")) \
    .withColumn("long", col("long").cast("double"))

# Проверка
trips.printSchema()
stations.printSchema()

root
 |-- id: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: string (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)



In [5]:
trips.show(5)
stations.show(5)

+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|8/29/2013 14:13|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|8/29/2013 10:16|Mountain View Cit...|              27|8/29/2013 10:17|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|8/29/2013 11:29|  San Jose City Hall|      

## 1. Найти велосипед с максимальным временем пробега.

In [11]:
from pyspark.sql.functions import sum as _sum

max_bike = trips \
    .groupBy("bike_id") \
    .agg(_sum("duration").alias("total_duration")) \
    .orderBy(col("total_duration").desc()) \
    .limit(1)

max_bike.show()

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
+-------+--------------+



## 2. Найти наибольшее геодезическое расстояние между станциями.

In [10]:
from pyspark.sql.functions import radians, sin, cos, sqrt, asin

# Две копии таблицы станций
s1 = stations.alias("s1")
s2 = stations.alias("s2")

# Все уникальные пары станций
station_pairs = s1.join(
    s2,
    col("s1.id") < col("s2.id")
)

# Радиус Земли в километрах
R = 6371.0

# Формула гаверсинуса
result = station_pairs.select(
    col("s1.id").alias("station_1_id"),
    col("s1.name").alias("station_1_name"),
    col("s2.id").alias("station_2_id"),
    col("s2.name").alias("station_2_name"),
    (
        2 * R * asin(
            sqrt(
                sin((radians(col("s2.lat")) - radians(col("s1.lat"))) / 2) ** 2 +
                cos(radians(col("s1.lat"))) * cos(radians(col("s2.lat"))) *
                sin((radians(col("s2.long")) - radians(col("s1.long"))) / 2) ** 2
            )
        )
    ).alias("distance_km")
)

result.orderBy(col("distance_km").desc()).show(1, truncate=False)

+------------+--------------------------+------------+----------------------+-----------------+
|station_1_id|station_1_name            |station_2_id|station_2_name        |distance_km      |
+------------+--------------------------+------------+----------------------+-----------------+
|16          |SJSU - San Salvador at 9th|60          |Embarcadero at Sansome|69.92087595428183|
+------------+--------------------------+------------+----------------------+-----------------+
only showing top 1 row


## 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [16]:
# Рез из задания 1
bike_id_value = max_bike.collect()[0]["bike_id"]

# все поездки этого велосипеда
bike_trips = trips.filter(col("bike_id") == bike_id_value)

#сортировка по времени
bike_trips = bike_trips.orderBy(col("start_date"))

# вывод пути
bike_trips.select(
    "start_date",
    "start_station_name",
    "end_station_name",
    "duration"
).show(50, truncate=False)

+---------------+---------------------------------------------+---------------------------------------------+--------+
|start_date     |start_station_name                           |end_station_name                             |duration|
+---------------+---------------------------------------------+---------------------------------------------+--------+
|1/1/2014 13:42 |Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |3289    |
|1/1/2014 18:51 |Embarcadero at Sansome                       |Market at 4th                                |1286    |
|1/1/2014 19:48 |Market at 4th                                |South Van Ness at Market                     |795     |
|1/10/2014 20:13|Market at 10th                               |Powell Street BART                           |235     |
|1/10/2014 8:09 |Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)     |596     |
|1/10/2014 8:21 |San Francisco Caltrain (Townsen

## 4. Найти количество велосипедов в системе.

In [17]:
trips.select("bike_id").distinct().count()

700

## 5. Найти пользователей потративших на поездки более 3 часов.

In [19]:
result = trips \
    .groupBy("zip_code") \
    .agg(_sum("duration").alias("total_duration")) \
    .filter(col("total_duration") > 10800)

result.show()

+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|   94102|      19128021|
|   95134|        728023|
|   84606|         95145|
|   80305|        180906|
|   60070|         28919|
|   95519|         30303|
|   43085|         11670|
|   91910|         50488|
|   77339|         13713|
|   48063|         13755|
|   85022|         12682|
|    1090|         20391|
|    2136|         16010|
|   11722|         24331|
|   95138|        155365|
|   94610|       3630628|
|   94404|       3589350|
|   80301|        152189|
|   91326|         65885|
|   90742|         10965|
+--------+--------------+
only showing top 20 rows
